# 02b — Full single-cell preprocessing (GSE132465, all tumor cells)

Load all tumor (`-T_`) cells across all patients, run QC/clustering/annotation, and save `_full` outputs.


## Install dependencies


In [1]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scanpy', 'anndata', 'numpy', 'pandas', 'scipy', 'matplotlib', 'scikit-learn', 'python-igraph', 'leidenalg'])


0

## Setup and reproducibility


In [2]:
from pathlib import Path
import os, sys, json, time, random
import numpy as np
import pandas as pd

USE_DRIVE = True
SEED = 42

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/BulkCellGNN_data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, '/content/drive/MyDrive/BulkCellGNN_data')

random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)

print('DATA_DIR =', DATA_DIR)

import scanpy as sc
import matplotlib.pyplot as plt

try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
except Exception:
    torch = None

sc.settings.verbosity = 2
try:
    sc.settings.seed = int(SEED)
except Exception:
    pass
device = torch.device('cuda' if (torch is not None and torch.cuda.is_available()) else 'cpu') if torch is not None else 'cpu'
print('device =', device)


Mounted at /content/drive
DATA_DIR = /content/drive/MyDrive/BulkCellGNN_data
device = cuda


## Read header and select tumor columns


In [3]:
import gzip
MATRIX_FILE = DATA_DIR / 'GSE132465_GEO_processed_CRC_10X_raw_UMI_count_matrix.txt.gz'
if not MATRIX_FILE.exists():
    raise FileNotFoundError(f'Missing matrix file: {MATRIX_FILE}')

with gzip.open(MATRIX_FILE, 'rt', encoding='utf-8') as f:
    header_line = f.readline().strip()
all_barcodes = header_line.split('\t')

tumor_cols = [i for i, b in enumerate(all_barcodes) if '-T_' in b]
normal_cols = [i for i, b in enumerate(all_barcodes) if '-N_' in b]
print('Total columns:', len(all_barcodes))
print('Tumor barcodes:', len(tumor_cols))
print('Normal barcodes available:', len(normal_cols))
if not tumor_cols:
    raise RuntimeError('No tumor barcodes found')


Total columns: 63690
Tumor barcodes: 47285
Normal barcodes available: 16404


## Load tumor-only matrix with usecols


In [4]:
t0 = time.time()
gene_col_idx = 0
cols_to_load = [gene_col_idx] + tumor_cols
print(f'Reading {len(tumor_cols):,} tumor columns...')
df = pd.read_csv(MATRIX_FILE, sep='\t', compression='gzip', usecols=cols_to_load, index_col=0, low_memory=False)
print('Loaded genes x cells:', df.shape, '| elapsed_sec=', round(time.time() - t0, 2))
print('Memory before cast:', df.memory_usage(deep=True).sum() / 1e9, 'GB')

# Cast to float32 after loading — data columns are now safe to convert
df = df.astype(np.float32)
print('Memory after cast:', df.memory_usage(deep=True).sum() / 1e9, 'GB')

adata = sc.AnnData(
    X=df.values.T.astype(np.float32),
    obs=pd.DataFrame(index=df.columns.astype(str)),
    var=pd.DataFrame(index=df.index.astype(str))
)
adata.var_names_make_unique()
del df
print('AnnData cells x genes:', adata.shape)


Reading 47,285 tumor columns...
Loaded genes x cells: (33694, 47285) | elapsed_sec= 204.55
Memory before cast: 12.747685534 GB
Memory after cast: 6.374802374 GB
AnnData cells x genes: (47285, 33694)


## Save patient IDs and optional normal barcode metadata


In [5]:
def patient_id_from_barcode(b):
    return str(b).split('-')[0]

patient_ids = np.array([patient_id_from_barcode(b) for b in adata.obs_names], dtype=object)
adata.obs['patient_id'] = patient_ids
np.save(DATA_DIR / 'cell_patient_ids_full.npy', patient_ids)

normal_barcodes = [all_barcodes[i] for i in normal_cols]
(DATA_DIR / 'normal_barcodes_available.json').write_text(json.dumps(normal_barcodes[:2000]), encoding='utf-8')
print('Saved cell_patient_ids_full.npy')


Saved cell_patient_ids_full.npy


## QC + normalization + HVGs


In [6]:
t0 = time.time()
adata.var['mt'] = adata.var_names.str.upper().str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)

print('Before QC:', adata.shape)
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
mt_col = 'pct_counts_mt' if 'pct_counts_mt' in adata.obs.columns else next((c for c in adata.obs.columns if c.startswith('pct_counts_') and 'mt' in c.lower()), None)
if mt_col is not None:
    adata = adata[adata.obs[mt_col] < 20].copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=5000, flavor='seurat', subset=True)
print('After QC + HVG:', adata.shape, '| elapsed_sec=', round(time.time() - t0, 2))


Before QC: (47285, 33694)
filtered out 8626 genes that are detected in less than 3 cells
normalizing counts per cell
    finished (0:00:00)
extracting highly variable genes
    finished (0:00:05)
After QC + HVG: (47285, 5000) | elapsed_sec= 69.19


## PCA / neighbors / UMAP / Leiden


In [7]:
t0 = time.time()
sc.tl.pca(adata, n_comps=50, svd_solver='arpack', random_state=SEED)
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=50, random_state=SEED)
sc.tl.umap(adata, random_state=SEED)
sc.tl.leiden(adata, resolution=0.5, random_state=SEED)
print('Computed manifold + Leiden | elapsed_sec=', round(time.time() - t0, 2))


computing PCA
    with n_comps=50
    finished (0:00:10)
computing neighbors
    using 'X_pca' with n_pcs = 50
    finished (0:01:10)
computing UMAP
    finished (0:00:39)
running Leiden clustering


/tmp/ipykernel_18812/3519805.py:5: FutureWarning: The `igraph` implementation of leiden clustering is *orders of magnitude faster*. Set the flavor argument to (and install if needed) 'igraph' to use it.
In the future, the default backend for leiden will be igraph instead of leidenalg. To achieve the future defaults please pass: `flavor='igraph'` and `n_iterations=2`. `directed` must also be `False` to work with igraph’s implementation.
  sc.tl.leiden(adata, resolution=0.5, random_state=SEED)


    finished (0:00:32)
Computed manifold + Leiden | elapsed_sec= 152.63


## Marker-based annotation


In [8]:
markers = {
    'T_cell': ['CD3D', 'CD3E'],
    'B_cell': ['CD19', 'MS4A1'],
    'Myeloid': ['LYZ', 'CD68'],
    'Epithelial': ['EPCAM', 'KRT8'],
    'Stromal': ['DCN', 'COL1A1'],
    'NK': ['NKG7', 'GNLY'],
}
present = {k: [g for g in genes if g in adata.var_names] for k, genes in markers.items()}

def label_cluster(cluster_id: str) -> str:
    sub = adata[adata.obs['leiden'] == cluster_id]
    scores = {}
    for ct, genes in present.items():
        if not genes:
            scores[ct] = -1e9
            continue
        X_sub = sub[:, genes].X
        if hasattr(X_sub, 'toarray'):
            X_sub = X_sub.toarray()
        scores[ct] = float(np.asarray(X_sub, dtype=np.float32).mean())
    return max(scores, key=scores.get)

clusters = sorted(adata.obs['leiden'].unique(), key=lambda x: (len(str(x)), str(x)))
map_leiden_to_type = {c: label_cluster(c) for c in clusters}
adata.obs['cell_type_str'] = adata.obs['leiden'].map(map_leiden_to_type)

type_names = sorted(adata.obs['cell_type_str'].unique())
type_to_idx = {n: i for i, n in enumerate(type_names)}
adata.obs['cell_type'] = adata.obs['cell_type_str'].map(type_to_idx).astype(int)
print('Cell types:', type_names)


Cell types: ['B_cell', 'Myeloid', 'NK', 'Stromal', 'T_cell']


## UMAP colored by patient ID


In [9]:
fig = sc.pl.umap(adata, color=['patient_id'], return_fig=True, show=False)
fig.savefig(DATA_DIR / 'umap_by_patient_full.png', dpi=200, facecolor='white', bbox_inches='tight')
plt.close(fig)
print('Saved umap_by_patient_full.png')


Saved umap_by_patient_full.png


## Save `_full` outputs


In [10]:
X = adata.X
if hasattr(X, 'toarray'):
    cell_expr_full = X.toarray().astype(np.float32)
else:
    cell_expr_full = np.asarray(X, dtype=np.float32)

cell_genes_full = np.asarray(adata.var_names.astype(str))
cell_types_full = adata.obs['cell_type'].to_numpy(dtype=np.int64)
cell_umap_full = adata.obsm['X_umap'].astype(np.float32)
cell_patient_ids_full = adata.obs['patient_id'].to_numpy()

np.save(DATA_DIR / 'cell_expr_full.npy', cell_expr_full)
np.save(DATA_DIR / 'cell_genes_full.npy', cell_genes_full)
np.save(DATA_DIR / 'cell_types_full.npy', cell_types_full)
np.save(DATA_DIR / 'cell_umap_full.npy', cell_umap_full)
np.save(DATA_DIR / 'cell_patient_ids_full.npy', cell_patient_ids_full)
(DATA_DIR / 'cell_type_names_full.json').write_text(json.dumps(type_names), encoding='utf-8')
print('Saved full outputs')


Saved full outputs


## Final verification


In [11]:
expected = ['cell_expr_full.npy','cell_genes_full.npy','cell_types_full.npy','cell_umap_full.npy','cell_type_names_full.json','cell_patient_ids_full.npy','umap_by_patient_full.png']
missing = []
for name in expected:
    p = DATA_DIR / name
    if p.exists():
        print('OK ', name)
    else:
        print('MISSING', name)
        missing.append(name)
if missing:
    raise FileNotFoundError('Missing outputs: ' + ', '.join(missing))
print('Notebook 02b complete')


OK  cell_expr_full.npy
OK  cell_genes_full.npy
OK  cell_types_full.npy
OK  cell_umap_full.npy
OK  cell_type_names_full.json
OK  cell_patient_ids_full.npy
OK  umap_by_patient_full.png
Notebook 02b complete
